<a href="https://colab.research.google.com/github/phoenix-fyre/victor-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phoenix-fyre/victor-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## Lane 3: Content Archetype Clustering

I am choosing Lane 3 to group the content inventory into archetypes using
structured search and engagement metrics (CTR, average position, engagement
rate, content type). Before committing to this lane, I checked the data
quality of these metrics rather than assuming they were clean.

A naive average CTR by content_type initially produced an impossible value
(feedly article mean CTR = 2.79, when CTR cannot exceed 1.0 by definition).
Investigating further, I found this wasn't isolated: 1,689 of 30,000 rows
(~5.6%) across the entire dataset have a CTR above 1.0, consistent with a
subset of values being recorded on a 0–100 percentage scale instead of a
0–1 decimal scale. This error is not evenly distributed — it affects
feedly articles at roughly 19.3% of that group, compared to ~4% for
comparison and keyword articles — which is itself a non-obvious finding
about how this content type's data is collected or processed. After
correcting values above 1.0 to their decimal-scale equivalent, the
feedly-article mean CTR dropped from 2.79 to a plausible 0.058.

This tells me two things that make Lane 3 a reasonable starting point:
first, that raw aggregate metrics in this dataset cannot be trusted at
face value, and clustering — done with proper scaling, outlier handling,
and cleaned inputs — is a method built to be more robust to exactly this
kind of noise than a simple rule or average would be. Second, that once
cleaned, content types already show real behavioral differences (e.g. the
position-tier CTR gradient from page_1 at 0.35 down to deep at 0.055),
which is the kind of structural variation archetype clustering is meant
to surface into actionable groups. I'm choosing this lane because the
evidence supports it — not because it seemed like the easiest option —
and I'm aware that further columns (avg_position, engagement_rate) will
need the same quality check before I trust any archetype built from them.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## Section 2: The Question — Decision, Action, Cost of a Wrong Call

For a content editor deciding what to work on today, we will build a
ranked table that tags each content item with an archetype, its key
metrics and values, and a suggested action (protect, rewrite, monitor,
or prune) — ranking items using structured signals such as CTR, average
position, and engagement rate. The action: an editor scans this list and
decides which pages to leave alone, which to rewrite, and which to
deprioritize. The unit of analysis is one content item (one row = one page)

A wrong call here has a real cost. If a page is misclassified — for
example, labeled as "underperforming" when it is actually a healthy,
well-ranking page — an editor might rewrite or alter content that
already ranks well, risking its existing search authority and causing
a measurable drop in its traffic. That's wasted editor time at best,
and self-inflicted damage to a working page at worst.

A plain rule is not enough because raw metrics in this dataset can carry
hidden errors that a simple threshold would trust blindly. I found this
directly: the raw CTR column contains a scaling bug affecting ~5.6% of
rows (spiking to 19.3% within feedly articles), where some values were
recorded on a 0–100 scale instead of 0–1. A rule like "CTR > 0.5 = top
performer" would have flagged these broken rows as strong performers
before I caught and corrected the error. Clustering, applied to properly
cleaned and scaled inputs, is more robust to this kind of noise than a
fixed if-statement rule would be — though it is not immune to it, which
is why the cleaning step came first.

We will claim only decision-support results: this output ranks and
labels candidates for human review, and does not claim to guarantee
that any recommended action (rewrite, prune, etc.) will improve a
page's performance.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd

# 1. Load the dataset first so Python knows what 'df' is
url = "https://raw.githubusercontent.com/phoenix-fyre/victor-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 2. Print your 2-3 key metrics
print("--- Content Types ---")
print(df['content_type'].value_counts())

print("\n--- Summary Averages ---")
print(df.groupby('content_type')[['ctr', 'avg_position', 'engagement_rate']].mean())

--- Content Types ---
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

--- Summary Averages ---
                         ctr  avg_position  engagement_rate
content_type                                               
comparison article  0.131205     11.296844         0.294118
feedly article      2.791274      5.771756         1.791450
keyword article     0.344766     17.285989         2.649161


In [4]:
df[df['content_type']=='feedly article']['ctr'].describe()

,ctr
count,2096.000000
mean,2.791274
std,9.358007
min,0.000000
25%,0.000000
50%,0.000000
75%,0.200000
max,100.000000


In [5]:
df[df['content_type']=='feedly article']['ctr'].value_counts(bins=10).sort_index()

,count
"(-0.101, 10.0]",1929
"(10.0, 20.0]",79
"(20.0, 30.0]",26
"(30.0, 40.0]",36
"(40.0, 50.0]",19
"(50.0, 60.0]",0
"(60.0, 70.0]",0
"(70.0, 80.0]",0
"(80.0, 90.0]",0
"(90.0, 100.0]",7


In [6]:
import pandas as pd

# 1. Load the dataset first so Python knows what 'df' is
url = "https://raw.githubusercontent.com/phoenix-fyre/victor-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 2. Fix the CTR scale inconsistency (convert values > 1.0 from percentage to decimal scale)
df['ctr_cleaned'] = df['ctr'].apply(lambda x: x / 100 if x > 1 else x)

# 3. Verify the fix worked
print("Original Max CTR:", df['ctr'].max())
print("Cleaned Max CTR:", round(df['ctr_cleaned'].max(), 4))
print("Number of rows fixed:", (df['ctr'] > 1).sum())

# 4. Check the updated mean for feedly articles
print("\nNew mean CTR for Feedly articles:", round(df[df['content_type'] == 'feedly article']['ctr_cleaned'].mean(), 4))

Original Max CTR: 100.0
Cleaned Max CTR: 1.0
Number of rows fixed: 1689

New mean CTR for Feedly articles: 0.0584


In [7]:
print(df.groupby('content_type').apply(lambda x: (x['ctr'] > 1).sum()))

content_type
comparison article      28
feedly article         405
keyword article       1256
dtype: int64


/tmp/ipykernel_41392/3951098784.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df.groupby('content_type').apply(lambda x: (x['ctr'] > 1).sum()))


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## Section 4: Careful Words — What I Can and Can't Claim

**What I can claim:** Once clusters are built, I can honestly say they show
groupings based on the structured signals available in this dataset (CTR,
average position, engagement rate) — a pattern the data supports, not a
definitive "true type" for each page. The archetype labels are a useful
lens for prioritizing review, not an objective classification.

**What I cannot claim:** My output will provide suggested actions and
decision-support results, not proven outcomes. I cannot claim that acting
on a cluster's suggested action (e.g. rewriting a "low-CTR, high-impression"
page) will definitely improve that page's performance — proving that would
require an actual experiment (e.g. A/B testing a rewrite), which this data
alone cannot provide. I also cannot claim to have discovered the "correct"
or only valid way to group this content — a different clustering method or
feature set could reasonably produce different, equally defensible groupings.

**On the CTR correction specifically:** The scaling fix I applied (dividing
values above 1.0 by 100) is my most plausible inference for what caused the
error, based on the fact that CTR cannot physically exceed 1.0 — not a
verified fact about how the error originated. I'm treating it as a
reasonable correction, not a confirmed ground truth.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.